In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import Dense, SimpleRNN, LSTM, Embedding, Dropout
from tensorflow.keras.models import Sequential
import tensorflow_datasets as tfds
import pprint

In [201]:
config = {
    "data_path": "shakespeare.txt",
    "sequence_length": 100,
    "batch_size": 64,
    "embedding_size": 256,
    "hidden_size": 512,
    "learning_rate": 0.001,
    "epochs": 50,
    "dropout": 0.2,
}


In [202]:
with open(config["data_path"], "r") as f:
    text = f.read()

print("Text length:", len(text))
print("Text sample:", text[:100])

Text length: 1115394
Text sample: First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [203]:
vocab = sorted(set(text))
vocab_size = len(vocab)
print(f"{vocab_size} unique characters")

65 unique characters


In [204]:
char_to_idx = {c: i for i, c in enumerate(vocab)}
idx_to_char = {i: c for i, c in enumerate(vocab)}
text_as_int = [char_to_idx[c] for c in text]

In [205]:
dataset = tf.data.Dataset.from_tensor_slices(text_as_int)

# Create sequences
sequences = dataset.batch(config["sequence_length"] + 1, drop_remainder=True)

def split_input_target(chunk):
    return chunk[:-1], chunk[1:]

dataset = sequences.map(split_input_target)

dataset = (
    dataset
    .shuffle(10000)
    .batch(config["batch_size"], drop_remainder=True)
    .prefetch(tf.data.AUTOTUNE)
)


In [ ]:
model = Sequential([
    Embedding(vocab_size, config["embedding_size"]),
    LSTM(
        config["hidden_size"],
        return_sequences=True,
    ),
    Dropout(config["dropout"]),
    LSTM(
        config["hidden_size"],
        return_sequences=True,
    ),
    Dropout(config["dropout"]),
    Dense(vocab_size) 
])


model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=config["learning_rate"],
        clipnorm=1.0
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
)


model.fit(dataset, epochs=config["epochs"])


Epoch 1/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 59s 337ms/step - loss: 2.7561
Epoch 2/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 57s 330ms/step - loss: 2.0186
Epoch 3/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 57s 331ms/step - loss: 1.7641
Epoch 4/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 56s 324ms/step - loss: 1.6191
Epoch 5/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 56s 324ms/step - loss: 1.5284
Epoch 6/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 56s 325ms/step - loss: 1.4657
Epoch 7/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 56s 325ms/step - loss: 1.4211
Epoch 8/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 58s 333ms/step - loss: 1.3851
Epoch 9/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 56s 324ms/step - loss: 1.3562
Epoch 10/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 56s 324ms/step - loss: 1.3307
Epoch 11/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 56s 324ms/step - loss: 1.3097
Epoch 12/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 56s 324ms/step - loss: 1.2899
Epoch 13/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 57s 329ms/step - loss: 1.2724
Epoch 14/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 56s 325ms/step - loss: 1.2556
Epoch 15/50
172

In [207]:
model.save('char_based_text_generation.keras')

In [210]:
def top_k_logits(logits, k=10):
    values, indices = tf.math.top_k(logits, k)
    min_values = values[:, -1, tf.newaxis]
    return tf.where(
        logits < min_values,
        tf.ones_like(logits) * -1e10,
        logits
    )
def generate_text(
    model,
    start_string,
    num_generate=500,
    temperature=0.5,
    top_k=10
):
    input_ids = [char_to_idx[c] for c in start_string]
    input_ids = tf.expand_dims(input_ids, 0)

    text_generated = []

    for _ in range(num_generate):
        input_context = input_ids[:, -config["sequence_length"]:]
        logits = model(input_context)
        logits = logits[:, -1, :] 

        logits /= temperature
        logits = top_k_logits(logits, top_k)

        predicted_id = tf.random.categorical(logits, 1)[0, 0].numpy()

        text_generated.append(idx_to_char[predicted_id])
        input_ids = tf.concat([input_ids, [[predicted_id]]], axis=-1)

    return start_string + "".join(text_generated)
print(
    generate_text(
        model,
        start_string="lets lets go to the ",
        num_generate=1000,
        temperature=0.5,
        top_k=10
    )
)


lets lets go to the ground,
I would proclaim you, if they should be so still
Against the father with the heart of sorrow.

CAPULET:
Well, let it straight against the sea that fair
To come to the and throat, that deadly good more
Than the absent.

COMINIUS:
I thank you. Lords of Lancaster.

GLOUCESTER:
And there too much of this?

FLORIZEL:
What art thou, which I must be thus?
Call them again with him! Come, good my lord;
But when the sea with his throat to the sun
And for some spirits of the sea-side at the field
Of this account the golden play of them.

KING RICHARD II:
Why, then, the king and height of my soul,
Were well I wot a man, and we his life,
That I have said 'twas the more stretch the foe.

KING RICHARD II:
A pair of such deep sister with the field
And threat the glory of the slavish of my soul,
As thou art a table to the hate of the world
That thou hast spoke to die: and therefore say,
Resolve me here, sir, in a part, though it should not
see the foul parties, and so shrive